In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/datadrivendecision.csv')
df = df.drop(columns=['Unnamed: 4', 'Unnamed: 11', 'country', 'city', 'target_uni'], errors='ignore')
df.head()

,university,distance_gnv_km,avg_temp,annual_precipitation,quality_life_index,per_sem_costs_chf,avg_livingcosts_chf,composite_reputation_index
0,University of St. Gallen (HSG),281,12.0,1031.0,210.64,1310,1419.9,86
1,Université de Lausanne (HEC Lausanne),51,14.7,1117.0,193.46,500,1404.2,70
2,Universität Zürich,224,13.9,1140.0,197.02,720,1448.6,88
3,London School of Economics (LSE),747,13.7,558.0,168.24,15600,1123.2,100
4,University of Edinburgh,1262,11.3,1172.0,191.77,5325,961.4,72


In [ ]:
df['avg_livingcosts_chf'] = df['avg_livingcosts_chf'] * 0.6

In [ ]:
#assigning points to the features
points = {
    'distance_gnv_km':          11,
    'avg_temp':                 5,
    'annual_precipitation':     7,
    'quality_life_index':      16,
    'per_sem_costs_chf':        13,
    'avg_livingcosts_chf':      26,
    'composite_reputation_index':22
}

#Normalization
total_points = sum(points.values())
weights = {feature: p / total_points for feature, p in points.items()}
weights

{'distance_gnv_km': 0.11,
 'avg_temp': 0.05,
 'annual_precipitation': 0.07,
 'quality_life_index': 0.16,
 'per_sem_costs_chf': 0.13,
 'avg_livingcosts_chf': 0.26,
 'composite_reputation_index': 0.22}

In [ ]:

benefit_features = ['avg_temp', 'quality_life_index', 'composite_reputation_index']
cost_features    = ['distance_gnv_km', 'annual_precipitation',
                    'per_sem_costs_chf', 'avg_livingcosts_chf']

# Min-max scaling
for col in benefit_features:
    col_min, col_max = df[col].min(), df[col].max()
    df[col + '_norm'] = (df[col] - col_min) / (col_max - col_min)

for col in cost_features:
    col_min, col_max = df[col].min(), df[col].max()
    df[col + '_norm'] = (col_max - df[col]) / (col_max - col_min)


In [ ]:
#Computation of weighted score for each university
df['overall_score'] = 0

for feature, w in weights.items():
    norm_col = feature + '_norm'
    df['overall_score'] += df[norm_col] * w

df = df.sort_values('overall_score', ascending=False).reset_index(drop=True)
df['rank'] = np.arange(1, len(df) + 1)

df[['rank', 'university', 'overall_score']]


,rank,university,overall_score
0,1,Erasmus University Rotterdam,0.662354
1,2,IE University Madrid,0.658683
2,3,University of Barcelona,0.652130
3,4,Humboldt University of Berlin,0.637761
4,5,University of Amsterdam,0.611326
5,6,HEC Paris,0.597844
6,7,University of Bologna,0.578869
7,8,Bocconi University,0.577981
8,9,University of St. Gallen (HSG),0.558708
9,10,London School of Economics (LSE),0.553568


In [ ]:

# Build clean ranking table
ranking_table = df[['rank', 'university', 'overall_score']].copy()
ranking_table.rename(
    columns={'rank': 'Rank', 'university': 'University', 'overall_score': 'Score'},
    inplace=True
)
ranking_table['Score'] = ranking_table['Score'].round(3)

# Style with full black background
styled_ranking = (
    ranking_table.style
    .format({'Score': '{:.3f}'})
    .set_properties(
        subset=['Rank', 'University', 'Score'],
        **{
            'background-color': 'black',
            'color': 'white',
            'text-align': 'center',
            'border': '1px solid white'
        }
    )
    .set_table_styles(
        [
            # Header style
            {'selector': 'th',
             'props': [('background-color', 'black'),
                       ('color', 'white'),
                       ('text-align', 'center'),
                       ('font-weight', 'bold'),
                       ('border', '1px solid white')]},

            # Remove index visibility (Colab workaround)
            {'selector': '.row_heading', 'props': [('display', 'none')]},
            {'selector': '.blank', 'props': [('display', 'none')]},
        ],
        overwrite=False
    )
)

styled_ranking



,Rank,University,Score
0,1,Erasmus University Rotterdam,0.662
1,2,IE University Madrid,0.659
2,3,University of Barcelona,0.652
3,4,Humboldt University of Berlin,0.638
4,5,University of Amsterdam,0.611
5,6,HEC Paris,0.598
6,7,University of Bologna,0.579
7,8,Bocconi University,0.578
8,9,University of St. Gallen (HSG),0.559
9,10,London School of Economics (LSE),0.554
